# 06 · Integración gold
**Responsable:** P6 · **Código:** `src/gold_integracion.py` · **Salidas:** `gold/municipio_anio.parquet`, `gold/maestro_boyaca.geoparquet`

**Qué hace**
- Parte de la población (123 municipios × 6 años) y le une con LEFT JOIN: el desempeño del bloque oficial (solo 2020), los totales agrícolas (solo 2023), el % rural y el resumen de suelo.
- Cada fuente llega ya agregada a su grano, así que una unión no puede duplicar filas. Si el número de filas cambia o la población total cambia, el pipeline se **detiene**.
- En cada unión registra cuántas claves coinciden y guarda las no vinculadas en `outputs/quality/no_vinculados_<fuente>.csv`.
- Calcula los indicadores: las 4 proporciones demográficas y la producción por cada 1.000 habitantes de 18–64 años. Una división entre 0 da null.
- `maestro_boyaca`: una fila por municipio con las variables de 2020 y de 2023 más la geometría, para los mapas.

**Por qué años distintos:** el desempeño solo existe para 2020 y la producción solo para 2023. Cada variable vive en su año y en los demás queda null, nunca 0.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [ ]:
import geopandas as gpd
from src.gold_agricola import resumen_municipal
from src.gold_integracion import construir_maestro, construir_municipio_anio
from src.silver_poblacion import pct_rural

pob = pl.read_parquet(SILVER / "poblacion.parquet")
areas = pl.read_parquet(SILVER / "poblacion_area.parquet")
mdm = pl.read_parquet(SILVER / "desempeno.parquet")
cm = pl.read_parquet(GOLD / "cultivo_municipio.parquet")
suelo_res = pl.read_parquet(GOLD / "suelo_resumen.parquet")
mun = gpd.read_parquet(SILVER / "municipios.geoparquet")

In [ ]:
ma = construir_municipio_anio(pob, mdm, resumen_municipal(cm), pct_rural(areas), suelo_res)
ma.group_by("anio").agg(pl.len(), pl.col("puntaje_mdm").count(), pl.col("produccion_total_t").count()).sort("anio")

In [ ]:
maestro = construir_maestro(ma, mun)
maestro.head()

In [ ]:
from src import graficos
graficos.distribucion_rural(ma.to_pandas(), ANIO_MDM)        # P1
graficos.produccion_por_1000(ma.to_pandas(), ANIO_EVA)        # P2

In [ ]:
pl.read_csv(QUALITY / "metricas_gold.csv")